# Crowding Robustness — Open Images v7 Pipeline
ResNet-34/50/101 · ViT-S/16 · ViT-B/16 · Swin-T · Swin-S · VGG KAGN-11v2 · VGG KAGN-11v4 · VGG KAGN-BN-SA-11v4

## 0. GPU Kontrolu

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Repo Klonla

In [ ]:
import os
REPO_URL = 'https://github.com/YOUR_USERNAME/crowding-robustness.git'
if not os.path.exists('/content/crowding-robustness'):
    os.system(f'git clone {REPO_URL} /content/crowding-robustness')
else:
    os.system('cd /content/crowding-robustness && git pull')
print('Repo hazir.')

## 2. Bagimliliklar

In [ ]:
import os
os.system('pip install -q torch torchvision timm fvcore fiftyone PyYAML tqdm opencv-python-headless seaborn pandas scikit-learn huggingface_hub safetensors')

# torch-conv-kan — sys.path'e EKLEME, importlib ile kullanilir
if not os.path.exists('/content/crowding-robustness/torch-conv-kan'):
    os.system('git clone https://github.com/IvanDrokin/torch-conv-kan.git /content/crowding-robustness/torch-conv-kan')
os.system('pip install -q -r /content/crowding-robustness/torch-conv-kan/requirements.txt')

# SAM2 — ayri konuma kur
if not os.path.exists('/content/sam2'):
    os.system('git clone https://github.com/facebookresearch/sam2.git /content/sam2')
    os.system('cd /content/sam2 && pip install -q -e .')

# SAM2 checkpoint
os.makedirs('/content/sam2_weights', exist_ok=True)
if not os.path.exists('/content/sam2_weights/sam2.1_hiera_small.pt'):
    os.system('wget -q -P /content/sam2_weights https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt')

print('Tum kurulumlar tamamlandi.')

## 3. Python Path Ayarla (Her session basinda calistir)

In [ ]:
import os, sys

os.chdir('/content/crowding-robustness')

# KRITIK: torch-conv-kan ve sam2 path'e EKLENMEMELI
# Her ikisinin de 'models/' ve 'sam2/' dizinleri bizimkiyle catisiyor
sys.path = [p for p in sys.path
            if 'torch-conv-kan' not in p
            and '/sam2' not in p
            and p != '/content']

if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

# Eski module cache temizle
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['models', 'data', 'train', 'evaluate', 'sam2']):
        del sys.modules[mod]

# __init__.py garantisi
for d in ['models', 'data', 'data/openimages', 'data/coco']:
    init = f'/content/crowding-robustness/{d}/__init__.py'
    os.makedirs(os.path.dirname(init), exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').close()

# Import testi
from models.resnet import build_resnet
from models.vit import build_vit
print('Importlar OK')

## 4. Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/crowding_openimages'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive hazir:', DRIVE_DIR)

## 5. Open Images v7 Indir (Sinif basina dengeli)

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from collections import Counter

TARGET_CLASSES = [
    'Car', 'Person', 'Bottle', 'Dog', 'Airplane',
    'Bird', 'Motorcycle', 'Horse', 'Cat', 'Book',
    'Truck', 'Bus', 'Laptop', 'Cake', 'Elephant',
    'Backpack', 'Sheep', 'Couch', 'Pizza', 'Toilet',
    'Clock', 'Giraffe', 'Zebra', 'Bear', 'Tiger',
    'Traffic light', 'Stop sign', 'Fire hydrant',
    'Skateboard', 'Suitcase'
]
MAX_PER_CLASS = 500

if fo.dataset_exists('crowding_openimages'):
    print('Dataset zaten mevcut, yukleniyor...')
    dataset = fo.load_dataset('crowding_openimages')
    print(f'Toplam: {len(dataset)} sample')
else:
    print('Her sinif icin ayri ayri indiriliyor...')
    tmp_names = []

    for cls in TARGET_CLASSES:
        tmp_name = f"tmp_{cls.replace(' ', '_').lower()}"
        if fo.dataset_exists(tmp_name):
            fo.delete_dataset(tmp_name)
        try:
            ds = foz.load_zoo_dataset(
                'open-images-v7',
                split='train',
                label_types=['segmentations'],
                classes=[cls],
                max_samples=MAX_PER_CLASS,
                dataset_name=tmp_name,
            )
            print(f'  {cls}: {len(ds)} sample')
            tmp_names.append(tmp_name)
        except Exception as e:
            print(f'  [WARN] {cls}: {e}')

    # Birlestir
    merged = fo.Dataset('crowding_openimages')
    merged.persistent = True
    for tmp_name in tmp_names:
        tmp_ds = fo.load_dataset(tmp_name)
        merged.add_samples(tmp_ds)
        fo.delete_dataset(tmp_name)

    dataset = merged
    print(f'\nToplam: {len(dataset)} sample')

# Sinif bazli kontrol
counts = Counter()
for sample in dataset.iter_samples():
    if sample.ground_truth is None:
        continue
    for det in sample.ground_truth.detections:
        if det.mask is not None and det.label in TARGET_CLASSES:
            counts[det.label] += 1

print('\nSinif bazli instance:')
for cls in TARGET_CLASSES:
    n = counts.get(cls, 0)
    status = 'OK' if n >= 50 else 'AZ'
    print(f'  [{status}] {cls}: {n}')

## 6. SAM2 ile Maske Iyilestir

In [ ]:
import os, sys, torch

# sam2 modüllerini temizle ve site-packages'tan yukle
for mod in list(sys.modules.keys()):
    if mod == 'sam2' or mod.startswith('sam2.'):
        del sys.modules[mod]
sys.path = [p for p in sys.path if '/sam2' not in p]
import site
for sp in site.getsitepackages():
    if sp not in sys.path:
        sys.path.insert(0, sp)

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam2_model = build_sam2(
    config_file='configs/sam2.1/sam2.1_hiera_s.yaml',
    ckpt_path='/content/sam2_weights/sam2.1_hiera_small.pt',
    device=device,
)
predictor = SAM2ImagePredictor(sam2_model)
print(f'SAM2 hazir ({device})')

In [ ]:
import sys, importlib

# models path catismasini onle
sys.path = [p for p in sys.path
            if 'torch-conv-kan' not in p
            and '/sam2' not in p
            and p != '/content']
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

import data.openimages.refine_masks as rm
importlib.reload(rm)
rm.main(
    'configs/config_openimages.yaml',
    sam2_config='configs/sam2.1/sam2.1_hiera_s.yaml',
    sam2_ckpt='/content/sam2_weights/sam2.1_hiera_small.pt'
)

## 7. Isolation Gorseller Olustur

In [ ]:
import sys, importlib
sys.path = [p for p in sys.path if 'torch-conv-kan' not in p and '/sam2' not in p and p != '/content']
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

import data.openimages.prepare_dataset as prep
importlib.reload(prep)
prep.main('configs/config_openimages.yaml')

In [ ]:
# Ornek isolation gorseller
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

dataset_dir = Path('./dataset_openimages')
all_imgs = list(dataset_dir.rglob('*.png'))
samples  = random.sample(all_imgs, min(8, len(all_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, p in zip(axes.flatten(), samples):
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{p.parts[-2]}\n{p.name}", fontsize=7)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 8. Crowding Kompozitleri Olustur

In [ ]:
import sys, importlib
sys.path = [p for p in sys.path if 'torch-conv-kan' not in p and '/sam2' not in p and p != '/content']
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

import data.openimages.build_composites as bc
importlib.reload(bc)
bc.main('configs/config_openimages.yaml')

In [ ]:
# Ornek kompozitler
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

comp_dir  = Path('./dataset_openimages/composites/test')
all_comps = list(comp_dir.rglob('*.png'))
samples   = random.sample(all_comps, min(8, len(all_comps)))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, p in zip(axes.flatten(), samples):
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{p.parts[-4]}/{p.parts[-3]}\n{p.parts[-2]}", fontsize=6)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Drive'a Kaydet (Ara kayit)

In [ ]:
import shutil, os
print('Dataset kaydediliyor...')
shutil.copytree('./dataset_openimages', f'{DRIVE_DIR}/dataset_openimages', dirs_exist_ok=True)
print('Tamamlandi.')

## 10. Model Egitimi (LP-FT)

In [ ]:
import sys, importlib
sys.path = [p for p in sys.path if 'torch-conv-kan' not in p and '/sam2' not in p and p != '/content']
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['models', 'data', 'train']):
        del sys.modules[mod]

import train as train_module

# Tek model
MODEL = 'resnet50'
sys.argv = ['train.py', '--model', MODEL, '--config', 'configs/config_openimages.yaml']
importlib.reload(train_module)
train_module.main()

In [ ]:
# Tum modeller
import sys, importlib
import train as train_module

MODELS = ['resnet34', 'resnet50', 'resnet101',
          'vit_s_16', 'vit_b_16', 'swin_t', 'swin_s',
          'vgg_kagn11_v2', 'vgg_kagn11_v4', 'vgg_kagn_bn11sa_v4']

for model_name in MODELS:
    print(f'\n=== {model_name} ===')
    sys.argv = ['train.py', '--model', model_name, '--config', 'configs/config_openimages.yaml']
    importlib.reload(train_module)
    train_module.main()

## 11. Degerlendirme

In [ ]:
import sys, importlib
sys.path = [p for p in sys.path if 'torch-conv-kan' not in p and '/sam2' not in p and p != '/content']
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

import evaluate as eval_module
sys.argv = ['evaluate.py', '--config', 'configs/config_openimages.yaml']
importlib.reload(eval_module)
eval_module.main()

## 12. Gorsellestirlmeler

In [ ]:
import sys, importlib
sys.path = [p for p in sys.path if 'torch-conv-kan' not in p and '/sam2' not in p and p != '/content']
if '/content/crowding-robustness' not in sys.path:
    sys.path.insert(0, '/content/crowding-robustness')

import visualize as viz_module
sys.argv = ['visualize.py', '--config', 'configs/config_openimages.yaml']
importlib.reload(viz_module)
viz_module.main()

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path('results_openimages/figures')
for fig_path in sorted(fig_dir.glob('*.png')):
    print(f'--- {fig_path.name} ---')
    display(Image(str(fig_path)))

## 13. Sonuclari Drive'a Kaydet

In [ ]:
import shutil
shutil.copytree('results_openimages', f'{DRIVE_DIR}/results_openimages', dirs_exist_ok=True)
print('Sonuclar kaydedildi.')